In [ ]:
import pandas as pd
import numpy as np
import uuid
import re
import json
import ast
from datetime import datetime

# ==========================================
# 1. CÁC HÀM HỖ TRỢ & TIỀN XỬ LÝ
# ==========================================
def make_slug(text):
    if not isinstance(text, str) or pd.isna(text): return np.nan
    text = str(text).lower()
    text = re.sub(r'[áàảãạăắằẳẵặâấầẩẫậ]', 'a', text)
    text = re.sub(r'[éèẻẽẹêếềểễệ]', 'e', text)
    text = re.sub(r'[íìỉĩị]', 'i', text)
    text = re.sub(r'[óòỏõọôốồổỗộơớờởỡợ]', 'o', text)
    text = re.sub(r'[úùủũụưứừửữự]', 'u', text)
    text = re.sub(r'[ýỳỷỹỵ]', 'y', text)
    text = re.sub(r'đ', 'd', text)
    text = re.sub(r'[^a-z0-9\s-]', '', text)
    text = re.sub(r'[\s-]+', '-', text).strip('-')
    return text

def parse_price(price_str):
    if pd.isna(price_str) or str(price_str).strip() == "": return np.nan
    clean_str = re.sub(r'[^\d]', '', str(price_str))
    return float(clean_str) if clean_str else np.nan

def format_specs(spec_str):
    if pd.isna(spec_str) or str(spec_str).strip() == '': return np.nan
    try:
        parsed = ast.literal_eval(str(spec_str))
        if isinstance(parsed, dict):
            formatted = [{"label": str(k).strip(), "value": str(v).strip()} for k, v in parsed.items()]
            return json.dumps(formatted, ensure_ascii=False)
    except: pass
    return np.nan

def clean_text_for_csv(text):
    if pd.isna(text): return np.nan
    clean_text = str(text).replace('\r\n', '\n').replace('\r', '\n')
    return clean_text.strip() if clean_text.strip() else np.nan

# HÀM MỚI: Gom 3 cột mô tả thành 1 bài hoàn chỉnh
def combine_descriptions(row):
    parts = []
    for col in ['Mô tả', 'Mô tả dài', 'Mô tả dài thành phần']:
        val = row.get(col)
        if pd.notna(val) and str(val).strip():
            parts.append(str(val).strip())

    if not parts: return np.nan
    return "\n\n".join(parts) # Nối các phần bằng 2 dấu xuống dòng cho dễ đọc

def fast_clean_base(name, brand):
    if pd.isna(brand) or not str(brand).strip(): return name
    first_word = str(brand).split()[0]
    match = re.search(re.escape(str(brand)), name, re.IGNORECASE)
    if not match: match = re.search(re.escape(first_word), name, re.IGNORECASE)

    if match:
        base = name[:match.end()].strip()
        if len(base.split()) <= 1:
            words = name[match.end():].strip().split()
            base = base + " " + " ".join(words[:2])
        return base.strip()
    return name

def parse_product_variant(full_name, base_name):
    variant_str = full_name[len(base_name):].strip() if full_name.startswith(base_name) else full_name
    variant_str = re.sub(r'^[\-\s,]+', '', variant_str).strip()

    size_pattern = r'(\d+(?:\.\d+)?\s*(?:ml|g|kg|l|oz|viên|miếng|thỏi|bản|bộ|gói|hộp))$'
    size_match = re.search(size_pattern, variant_str, re.IGNORECASE)
    size = size_match.group(1) if size_match else np.nan

    color = re.sub(size_pattern, '', variant_str, flags=re.IGNORECASE).strip()
    color = re.sub(r'[\-\s,]+$', '', color).strip()
    return color if color else np.nan, size

def extract_category_name(url):
    if pd.isna(url): return "Chưa phân loại"
    match = re.search(r'/danh-muc/([a-zA-Z0-9\-]+)-c\d+', str(url))
    return match.group(1).replace('-', ' ').title() if match else "Chưa phân loại"

# ==========================================
# 2. ĐỌC DỮ LIỆU
# ==========================================
print("Đang đọc dữ liệu gốc...")
try:
    df = pd.read_csv('hasaki_products_merged.csv', engine='python', on_bad_lines='skip', encoding='utf-8')
except Exception:
    df = pd.read_csv('hasaki_products_merged.csv', lineterminator='\n', encoding='utf-8')
    df.columns = df.columns.str.replace(r'\r', '', regex=True)

df['Tên sản phẩm'] = df['Tên sản phẩm'].astype(str).str.replace(r'\s*\[HSD.*?\]\s*', '', regex=True).str.strip()
df['GroupKey'] = df['Thương hiệu'].fillna('') + "||" + df['Mô tả dài'].fillna('')
df['Category_Name'] = df['Nguồn'].apply(extract_category_name)

now = datetime.now().isoformat()

# ==========================================
# 3. TẠO CÁC BẢNG THEO SCHEMA
# ==========================================
print("Đang xử lý dữ liệu và tạo bảng...")

brands = df[['Thương hiệu']].dropna().drop_duplicates().reset_index(drop=True)
brands.columns = ['name']
brands['id'] = [str(uuid.uuid4()) for _ in range(len(brands))]
brands['slug'] = brands['name'].apply(lambda x: f"{make_slug(x)}-{str(uuid.uuid4())[:4]}")
brands['createdAt'] = brands['updatedAt'] = now
brand_dict = dict(zip(brands['name'], brands['id']))
brands_df = brands[['id', 'name', 'slug', 'createdAt', 'updatedAt']]

categories = df[['Category_Name']].dropna().drop_duplicates().reset_index(drop=True)
categories.columns = ['name']
categories['id'] = [str(uuid.uuid4()) for _ in range(len(categories))]
categories['slug'] = categories['name'].apply(lambda x: f"{make_slug(x)}-{str(uuid.uuid4())[:4]}")
categories['createdAt'] = categories['updatedAt'] = now
category_dict = dict(zip(categories['name'], categories['id']))
categories_df = categories[['id', 'name', 'slug', 'createdAt', 'updatedAt']]

unique_images = df[['Ảnh']].dropna().drop_duplicates().reset_index(drop=True)
unique_images.columns = ['url']
unique_images['id'] = [str(uuid.uuid4()) for _ in range(len(unique_images))]
unique_images['createdAt'] = now
image_dict = dict(zip(unique_images['url'], unique_images['id']))
images_df = unique_images[['id', 'url', 'createdAt']]

products_data, variants_data, product_images_data = [], [], []

for key, group in df.groupby('GroupKey'):
    names = group['Tên sản phẩm'].tolist()
    if not names: continue

    first_row = group.iloc[0]
    base_name = fast_clean_base(names[0], first_row.get('Thương hiệu', ''))
    product_id = str(uuid.uuid4())

    sale_price = parse_price(first_row.get('Giá khuyến mãi', np.nan))
    price = parse_price(first_row.get('Giá gốc', np.nan))
    if pd.isna(price) and pd.notna(sale_price): price = sale_price

    valid_sale_price = sale_price if (pd.notna(sale_price) and pd.notna(price) and sale_price < price) else np.nan

    # LẤY MÔ TẢ DÀI ĐẦY ĐỦ TỪ 3 CỘT
    full_long_description = combine_descriptions(first_row)

    products_data.append({
        'id': product_id,
        'name': base_name,
        'slug': f"{make_slug(base_name)}-{str(uuid.uuid4())[:6]}",
        'short_description': clean_text_for_csv(first_row.get('Mô tả ngắn', np.nan)),
        'long_description': clean_text_for_csv(full_long_description),
        'status': 'ACTIVE',
        'price': price if pd.notna(price) else np.nan,
        'salePrice': valid_sale_price,
        'rating': 5.0,
        'sold': 0,
        'specifications': format_specs(first_row.get('Thông số kỹ thuật', np.nan)),
        'brandId': brand_dict.get(first_row.get('Thương hiệu'), np.nan),
        'categoryId': category_dict.get(first_row.get('Category_Name'), np.nan),
        'createdAt': now, 'updatedAt': now
    })

    first_img_url = first_row.get('Ảnh')
    if pd.notna(first_img_url) and first_img_url in image_dict:
        product_images_data.append({
            'id': str(uuid.uuid4()), 'productId': product_id,
            'imageId': image_dict[first_img_url], 'order': 0
        })

    for idx, row in group.iterrows():
        v_color, v_size = parse_product_variant(row['Tên sản phẩm'], base_name)
        img_id = image_dict.get(row.get('Ảnh'), np.nan)

        v_sale = parse_price(row.get('Giá khuyến mãi', np.nan))
        v_price = parse_price(row.get('Giá gốc', np.nan))
        if pd.isna(v_price) and pd.notna(v_sale): v_price = v_sale

        v_valid_sale = v_sale if (pd.notna(v_sale) and pd.notna(v_price) and v_sale < v_price) else np.nan

        v_cost = parse_price(row.get('Giá nhập', np.nan))
        if pd.isna(v_cost):
            base_p = v_sale if pd.notna(v_sale) else v_price
            v_cost = round((base_p * 0.6) / 1000) * 1000 if pd.notna(base_p) else np.nan

        variants_data.append({
            'id': str(uuid.uuid4()), 'productId': product_id,
            'color': v_color, 'size': v_size, 'imageId': img_id,
            'sku': f"SKU-{str(uuid.uuid4())[:8].upper()}",
            'price': v_price if pd.notna(v_price) else np.nan,
            'salePrice': v_valid_sale,
            'costPrice': v_cost,
            'statusName': 'NEW', 'status': 'ACTIVE',
            'createdAt': now, 'updatedAt': now
        })

products_df = pd.DataFrame(products_data)
variants_df = pd.DataFrame(variants_data)
product_images_df = pd.DataFrame(product_images_data)

# ==========================================
# 4. BỘ LỌC TÀN NHẪN (XÓA MỌI CỘT NULL TRỪ SOLD)
# ==========================================
print("Đang loại bỏ mọi dòng chứa Null/NaN...")

cols_to_check_prod = [col for col in products_df.columns if col != 'sold']
products_df = products_df.dropna(subset=cols_to_check_prod)

variants_df = variants_df.dropna()

# ==========================================
# 5. DỌN DẸP LẠI KHÓA NGOẠI
# ==========================================
valid_pids = set(products_df['id']).intersection(set(variants_df['productId']))
products_df = products_df[products_df['id'].isin(valid_pids)]
variants_df = variants_df[variants_df['productId'].isin(valid_pids)]

if not product_images_df.empty:
    product_images_df = product_images_df[product_images_df['productId'].isin(valid_pids)]
    product_images_df = product_images_df.drop_duplicates(subset=['productId', 'imageId'])

valid_bids = set(products_df['brandId'])
brands_df = brands_df[brands_df['id'].isin(valid_bids)]

valid_cids = set(products_df['categoryId'])
categories_df = categories_df[categories_df['id'].isin(valid_cids)]

valid_iids = set(variants_df['imageId'])
if not product_images_df.empty:
    valid_iids = valid_iids.union(set(product_images_df['imageId']))
images_df = images_df[images_df['id'].isin(valid_iids)]

# ==========================================
# 6. XUẤT FILE CSV
# ==========================================
print(f"Bắt đầu xuất {len(products_df)} Products và {len(variants_df)} Variants đặc ruột...")

brands_df.to_csv('Brand.csv', index=False)
categories_df.to_csv('Category.csv', index=False)
images_df.to_csv('Image.csv', index=False)
products_df.to_csv('Product.csv', index=False)
variants_df.to_csv('Variant.csv', index=False)

if not product_images_df.empty:
    product_images_df.to_csv('ProductImage.csv', index=False)

print("HOÀN TẤT XUẤT SẮC! Cột long_description giờ đã bao gồm tất cả nội dung chi tiết. 🚀")

Đang đọc dữ liệu gốc...
Đang xử lý dữ liệu và tạo bảng...
Đang loại bỏ mọi dòng chứa Null/NaN...
Bắt đầu xuất 862 Products và 1671 Variants đặc ruột...
HOÀN TẤT XUẤT SẮC! Cột long_description giờ đã bao gồm tất cả nội dung chi tiết. 🚀


In [1]:
import pandas as pd
import uuid
import re
from datetime import datetime

# 1. Đọc dữ liệu
hasaki = pd.read_csv('hasaki.csv')
brand_df = pd.read_csv('Brand (5).csv')
old_images = pd.read_csv('Image (4).csv')

# 2. Rút trích danh sách Brand, Logo, Banner từ hasaki.csv
records = []
suffixes = [""] + [f" {i}" for i in range(2, 200)]
for suffix in suffixes:
    name_col = f"text-sm{suffix}"
    logo_col = f"max-w-full src{suffix}"
    banner_col = f"max-w-[100%] src{suffix}"

    if name_col in hasaki.columns and logo_col in hasaki.columns and banner_col in hasaki.columns:
        temp = hasaki[[name_col, logo_col, banner_col]].copy()
        temp.columns = ['Brand', 'Logo', 'Banner']
        records.append(temp)

all_brands = pd.concat(records, ignore_index=True).dropna(subset=['Brand']).drop_duplicates(subset=['Brand'])

# Hàm làm sạch tên để so khớp chính xác
def clean_name(s):
    if not isinstance(s, str): return ""
    s = str(s).lower()
    return re.sub(r'[^a-z0-9]', '', s)

all_brands['name_clean'] = all_brands['Brand'].apply(clean_name)
brand_df['name_clean'] = brand_df['name'].apply(clean_name)

# 3. Quét ảnh: Tách riêng ảnh Cũ và ảnh Mới
image_map = dict(zip(old_images['url'], old_images['id']))
new_images_list = []
now = datetime.now().isoformat()

def get_or_create_image(url):
    if pd.isna(url) or str(url).strip() == "":
        return pd.NA

    # Nếu ảnh đã có trong DB -> Trả về ID cũ (Không tạo mới)
    if url in image_map:
        return image_map[url]

    # Nếu là ảnh mới tinh -> Tạo ID mới và đưa vào danh sách New Image
    new_id = str(uuid.uuid4())
    image_map[url] = new_id
    new_images_list.append({
        'id': new_id,
        'url': url,
        'createdAt': now
    })
    return new_id

# 4. So khớp và gắn Logo/Banner vào Brand
merged = pd.merge(brand_df, all_brands[['name_clean', 'Logo', 'Banner']], on='name_clean', how='left')

# Cột logoId và bannerId giờ sẽ chứa ID chuẩn xác (vừa cũ vừa mới)
merged['logoId'] = merged['Logo'].apply(get_or_create_image)
merged['bannerId'] = merged['Banner'].apply(get_or_create_image)

brand_cols = ['id', 'name', 'slug', 'logoId', 'bannerId', 'createdAt', 'updatedAt']
final_brand_df = merged[brand_cols].copy()

# 5. XUẤT FILE
new_images_df = pd.DataFrame(new_images_list)

if not new_images_df.empty:
    new_images_df.to_csv('Image_New_Only.csv', index=False)
    print(f"Thành công! Đã tạo Image_New_Only.csv chứa {len(new_images_df)} ảnh mới.")
else:
    print("Không có ảnh nào mới để thêm vào.")

final_brand_df.to_csv('Brand_Upsert.csv', index=False)
print(f"Đã tạo Brand_Upsert.csv chứa {len(final_brand_df)} thương hiệu.")

Thành công! Đã tạo Image_New_Only.csv chứa 328 ảnh mới.
Đã tạo Brand_Upsert.csv chứa 237 thương hiệu.


In [2]:
import pandas as pd
import uuid
import re
from datetime import datetime

# 1. Đọc dữ liệu
hasaki = pd.read_csv('hasaki.csv')
brand_df = pd.read_csv('Brand (5).csv')
old_images = pd.read_csv('Image (4).csv')

# 2. Rút trích danh sách Brand, Logo, Banner từ hasaki.csv
records = []
suffixes = [""] + [f" {i}" for i in range(2, 200)]
for suffix in suffixes:
    name_col = f"text-sm{suffix}"
    logo_col = f"max-w-full src{suffix}"
    banner_col = f"max-w-[100%] src{suffix}"

    if name_col in hasaki.columns and logo_col in hasaki.columns and banner_col in hasaki.columns:
        temp = hasaki[[name_col, logo_col, banner_col]].copy()
        temp.columns = ['Brand', 'Logo', 'Banner']
        records.append(temp)

all_brands = pd.concat(records, ignore_index=True).dropna(subset=['Brand']).drop_duplicates(subset=['Brand'])

def clean_name(s):
    if not isinstance(s, str): return ""
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

all_brands['name_clean'] = all_brands['Brand'].apply(clean_name)
brand_df['name_clean'] = brand_df['name'].apply(clean_name)

# 3. Quét ảnh để phân biệt cũ/mới
image_map = dict(zip(old_images['url'], old_images['id']))
new_images_list = []
now = datetime.now().isoformat()

def get_or_create_image(url):
    if pd.isna(url) or str(url).strip() == "":
        return None
    if url in image_map:
        return image_map[url]

    new_id = str(uuid.uuid4())
    image_map[url] = new_id
    new_images_list.append({
        'id': new_id,
        'url': url,
        'createdAt': now
    })
    return new_id

# Gắn Logo/Banner URL vào Brand DF để lấy ID
merged = pd.merge(brand_df, all_brands[['name_clean', 'Logo', 'Banner']], on='name_clean', how='left')
merged['logoId'] = merged['Logo'].apply(get_or_create_image)
merged['bannerId'] = merged['Banner'].apply(get_or_create_image)

# 4. Xuất file ảnh mới (Image_New_Only.csv)
new_images_df = pd.DataFrame(new_images_list)
if not new_images_df.empty:
    new_images_df.to_csv('Image_New_Only.csv', index=False)
    print(f"Đã tạo file 'Image_New_Only.csv' ({len(new_images_df)} ảnh mới).")

# 5. Xây dựng Script SQL UPDATE cho Brand
update_statements = []
for _, row in merged.iterrows():
    if pd.notna(row['logoId']) or pd.notna(row['bannerId']):
        logo_val = f"'{row['logoId']}'" if pd.notna(row['logoId']) else "NULL"
        banner_val = f"'{row['bannerId']}'" if pd.notna(row['bannerId']) else "NULL"

        stmt = f'UPDATE "Brand" SET "logoId" = {logo_val}, "bannerId" = {banner_val} WHERE "id" = \'{row["id"]}\';'
        update_statements.append(stmt)

# Lưu ra file SQL
with open('Update_Brand_Logos.sql', 'w', encoding='utf-8') as f:
    f.write("-- Script cập nhật Logo và Banner cho bảng Brand\n")
    f.write("\n".join(update_statements))

print(f"Đã tạo file 'Update_Brand_Logos.sql' ({len(update_statements)} lệnh UPDATE).")

Đã tạo file 'Image_New_Only.csv' (328 ảnh mới).
Đã tạo file 'Update_Brand_Logos.sql' (164 lệnh UPDATE).
